In [1]:

from MLProject import var_backtest_utils as vbu, var_lstm_xgboost as vlx
import plotly.graph_objects as go

# Project Overview


This project builds a framework for creating **Value-at-Risk (VaR)** forecasts with 3 different models and compares the performance of hyperparameter/ data actuality and density decisions.

# Feature and Target
Garchin return tahmini

$r(t) = μ + σ(t)·z(t)$ ile aciklanabilir,



$²(t) = ω + α·r(t-1)² + β·σ²(t-1): t$

(today's variance is a baseline plus a reaction to yesterday's shock plus persistence of yesterday's variance)

garch, ω/α/β ve μ degerlerini  MLE ile  predict ederek bu return degerini hesaplar.

z(t), r(t)'nin sistemin hesaplayamadigi kismini,

Bunu bir machine learning taska cevirmek icin,

target olarak



All four models plug into the same VaR pipeline: fit, predict
next-day variance, convert to VaR via the empirical quantile of standardized
residuals (inspired by calculating VaR via GARCH residuals), backtest with the Kupiec/Christoffersen/traffic-light test suite.

 Project is divided in 3 different files:
* `11_var_lstm_xgboost.py`  the modeling library: feature engineering and models
* `08_var_backtest_utils.py`  the backtest test suite
* 12_ml_models_benchmarking.py`  the benchmarking/tuning layer:
  which `training_window` and which hyperparameters produce the best-calibrated
  VaR forecasts

![prep_and_models.drawio.svg](archive/prep_and_models.drawio.svg)


## Feature Building
Currently, the framework only uses the closing price difference of tickers for each day. I'm planning to expend use multi day data, but as "hacky" guardrails implememented to check whether any date function is really inside the feature dataframe/ a weekday where trades happened needs general structural change to accept and clean single trading day with 6,5 days.
* (especially for GARCH based VaR forecast using multi day horizont, using the difference of

Target: squared percent return as a proxy for variance

* Portfolio weights can be rearranged via notebook wide definition of `WEIGHTS`. Currenttly an equal weight porfolio is assumed.

* According to goal, long or wide format features can be created.
    * if single tickers VaR  wanted to be predicted with available data `build_ml_features_portfolio_long_format` is used,
    * For portfolio wide var calculations 'build_ml_features_portfolio_wide_format' is used.
        * For tree models to capture the autocorrelative nature for volatility, lagged values of previous returns can be added.

For each day:
* return
* absolute_return
* in windows of 5,10 and 21 days (to capture short/meidum/long term effects)
    * realized volatility
    * mean return
    * SKEWNESS of return
        * to capture
* EWMA Volatility for 21 days
* laged values

are created as features




# Settings

In [3]:
import pandas as pd

In [4]:
DATA_PATH = "../data_general/4_ticker_data.csv"
TICKERS = ["AAPL", "JPM", "XOM", "KO"]
TICKERS_WITH_PORTFOLIO = TICKERS + ["PORTFOLIO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,

}

raw = pd.read_csv(DATA_PATH)

close_price = raw.copy()
close_price["Date"] = pd.to_datetime(close_price["Date"])
close_price = close_price.set_index("Date").sort_index()

percent_return_with_portfolio = close_price.pct_change()
percent_return_with_portfolio["PORTFOLIO"] = percent_return_with_portfolio[TICKERS].mul(WEIGHTS).sum(axis=1)
display(percent_return_with_portfolio.tail())

,AAPL,JPM,KO,XOM,PORTFOLIO
Date,,,,,
2024-12-24,0.011478,0.016444,0.007374,0.000941,0.009059
2024-12-26,0.003176,0.003425,-0.004297,0.000846,0.000788
2024-12-27,-0.013242,-0.008102,-0.001918,-0.000094,-0.005839
2024-12-30,-0.013263,-0.007671,-0.006725,-0.006762,-0.008605
2024-12-31,-0.007058,0.001630,0.003708,0.017114,0.003848


# Feature Creation

In [5]:
feature_single_ticker = vlx.build_ml_features_single_ticker(percent_return_with_portfolio["AAPL"])
display(feature_single_ticker.feature_cols)
display(feature_single_ticker.df.tail())

['r',
 'abs_r',
 'real_vol_5',
 'mean_abs_5',
 'skew_5',
 'real_vol_10',
 'mean_abs_10',
 'skew_10',
 'real_vol_21',
 'mean_abs_21',
 'skew_21',
 'ewma_vol']

,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,r_next,target
Date,,,,,,,,,,,,,,
2024-12-24,0.011478,0.011478,0.015260,0.012359,-1.424277,0.011168,0.009506,-1.347613,0.008772,0.008250,-1.376666,0.010001,0.003176,0.000010
2024-12-26,0.003176,0.003176,0.006617,0.008710,1.005256,0.010693,0.009307,-1.735579,0.008616,0.007779,-1.309222,0.009549,-0.013242,0.000175
2024-12-27,-0.013242,0.013242,0.011959,0.009955,-0.630557,0.012133,0.010035,-1.062820,0.009431,0.007962,-1.030182,0.010530,-0.013263,0.000176
2024-12-30,-0.013263,0.013263,0.011035,0.008845,-0.151149,0.013197,0.011292,-0.691947,0.010123,0.008567,-0.905319,0.011135,-0.007058,0.000050
2024-12-31,-0.007058,0.007058,0.010856,0.009643,0.702219,0.012947,0.010825,-0.291988,0.010245,0.008417,-0.681828,0.010919,NaN,NaN


###### Creating wide format features (for LSTM) without Lag

In [6]:
feature_wide_without_lag = vlx.build_ml_features_portfolio_wide_format(percent_return_with_portfolio)
feature_wide_without_lag.df.tail()


,AAPL,JPM,XOM,KO,PORTFOLIO,AAPL_abs_r,JPM_abs_r,XOM_abs_r,KO_abs_r,PORTFOLIO_abs_r,...,KO_skew_21,PORTFOLIO_skew_21,AAPL_ewma_vol,JPM_ewma_vol,XOM_ewma_vol,KO_ewma_vol,PORTFOLIO_ewma_vol,market_ewma_vol,r_next,target
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-24,0.011478,0.016444,0.000941,0.007374,0.009059,0.011478,0.016444,0.000941,0.007374,0.009059,...,0.109120,-1.056899,0.010001,0.015589,0.009229,0.009086,0.007843,0.007843,0.000788,6.202490e-07
2024-12-26,0.003176,0.003425,0.000846,-0.004297,0.000788,0.003176,0.003425,0.000846,0.004297,0.000788,...,0.277515,-1.032088,0.009549,0.014879,0.008927,0.008744,0.007479,0.007479,-0.005839,3.409459e-05
2024-12-27,-0.013242,-0.008102,-0.000094,-0.001918,-0.005839,0.013242,0.008102,0.000094,0.001918,0.005839,...,0.357047,-0.915466,0.010530,0.014456,0.008580,0.008346,0.007365,0.007365,-0.008605,7.405298e-05
2024-12-30,-0.013263,-0.007671,-0.006762,-0.006725,-0.008605,0.013263,0.007671,0.006762,0.006725,0.008605,...,0.419571,-0.751162,0.011135,0.013991,0.008239,0.008151,0.007450,0.007450,0.003848,1.481043e-05
2024-12-31,-0.007058,0.001630,0.017114,0.003708,0.003848,0.007058,0.001630,0.017114,0.003708,0.003848,...,0.393393,-0.751122,0.010919,0.013352,0.009974,0.007909,0.007241,0.007241,NaN,NaN


###### Creating wide long format with lag for tree based models

In [8]:
sq_return_lags =[1,2,3,10,20]
feature_long_with_lag = vlx.build_ml_features_portfolio_long_format(percent_return_with_portfolio, sq_return_lags=sq_return_lags)


In [9]:
feature_long_with_lag.df.tail()

,Date,ticker,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,...,skew_21,ewma_vol,r2_lag_1,r2_lag_2,r2_lag_3,r2_lag_10,r2_lag_20,r_next,target,market_ewma_vol
5027,2024-12-30,XOM,-0.006762,0.006762,0.003988,0.002541,-1.328193,0.008329,0.006501,-0.999001,...,-1.414383,0.008239,8.819000e-09,7.154214e-07,8.849144e-07,0.000077,6.500587e-06,0.017114,0.000293,0.007450
5028,2024-12-31,AAPL,-0.007058,0.007058,0.010856,0.009643,0.702219,0.012947,0.010825,-0.291988,...,-0.681828,0.010919,1.759202e-04,1.753561e-04,1.008533e-05,0.000138,9.067920e-05,NaN,NaN,0.007241
5029,2024-12-31,JPM,0.001630,0.001630,0.010034,0.007454,0.872998,0.015249,0.011037,-1.015830,...,-0.516830,0.013352,5.884300e-05,6.564672e-05,1.173264e-05,0.000002,1.930869e-04,NaN,NaN,0.007241
5030,2024-12-31,KO,0.003708,0.003708,0.005805,0.004804,0.451346,0.007023,0.005697,0.895839,...,0.393393,0.007909,4.523081e-05,3.678128e-06,1.846099e-05,0.000082,4.502909e-05,NaN,NaN,0.007241
5031,2024-12-31,XOM,0.017114,0.017114,0.008821,0.005151,1.448613,0.008627,0.006074,0.547632,...,-0.559216,0.009974,4.572224e-05,8.819000e-09,7.154214e-07,0.000457,8.695408e-07,NaN,NaN,0.007241


# Calling Wrapper Functions to Create Models Systematically

## Layer 1
![wrapper_layer_1.svg](archive/wrapper_layer_1.svg)

## Layer 2:

### Non-Rolling-Trainer
![non_rolling_trainer.svg](archive/non_rolling_trainer.svg)

In [10]:
WINDOW_SIZE = 250
PERIOD_OF_FORECAST = 252
VALIDATION_PERIOD = 20
N_TRIALS = 15

VAR_CONFIDENCE = 0.95
MODELS = ["garch", "xgboost", "random_forest", "lstm"]

In [11]:
len(feature_long_with_lag.df)

4948

## Test: forecasting the last 30 day in dataset via one trained model  for 252 and 100 days data with LSTM
The wrapper will remove the last 30days data from training set, and train a model with the last 100 and 252 days to predict daily VaR of our portfolio for 30 days for , and return the realized return/ var

In [12]:

thirty_day_var_with_100_day_data = vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(feature_wide_without_lag,"lstm",window_size=100,period_of_forecast=30,)

thirty_day_var_with_252_day_data = vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(feature_wide_without_lag,"lstm",window_size=252,period_of_forecast=30)

thirty_day_var_with_100_day_data.tail()


,varForecast,loss
2024-12-24,0.018488,-0.009059
2024-12-26,0.018739,-0.000788
2024-12-27,0.019215,0.005839
2024-12-30,0.017664,0.008605
2024-12-31,0.015126,-0.003848


In [13]:


fig = go.Figure()

fig.add_scatter(
    x=thirty_day_var_with_100_day_data.index,
    y=thirty_day_var_with_100_day_data["varForecast"],
    mode="lines",
    name="VaR Forecast (100-day window)",
)

fig.add_scatter(
    x=thirty_day_var_with_252_day_data.index,
    y=thirty_day_var_with_252_day_data["varForecast"],
    mode="lines",
    name="VaR Forecast (252-day window)",
)

fig.add_scatter(
    x=thirty_day_var_with_100_day_data.index,
    y=thirty_day_var_with_100_day_data["loss"],
    mode="lines",
    name="Loss",
)

fig.update_layout(
    title="LSTM: 30-Day VaR Forecast (100 vs 252-Day Window) and Loss",
    xaxis_title="Date",
    yaxis_title="Value",
    template="plotly_white",
)

fig.show()

In [14]:

vbu.run_full_backtest_suite_single_window(
    forecast_df=thirty_day_var_with_100_day_data,
    start_date=thirty_day_var_with_100_day_data.index[0],
    end_date=thirty_day_var_with_100_day_data.index[-1],
    var_confidence=0.95,
)

{'n_days': 30,
 'n_exceedances': 0,
 'exceedance_rate': 0.0,
 'expected_exceedance_rate': 0.050000000000000044,
 'binomial_pass': True,
 'traffic_light_zone': 1,
 'kupiec_pof_stat': np.float64(3.0775976632530346),
 'kupiec_pof_pass': True,
 'kupiec_tuff_pass': nan,
 'haas_tbf_pass': nan,
 'christoffersen_ind_stat': np.float64(-0.0),
 'christoffersen_ind_pass': True,
 'christoffersen_cc_stat': np.float64(3.0775976632530346),
 'christoffersen_cc_pass': True}

## Test: forecasting the last 30 day in dataset via one trained model  for 252 and 100 days data with Tree Based models

Using lagged and non-lagged versions

In [15]:
trained_models_results = {}

trained_models_results["xgboost_100"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_wide_without_lag,
        "xgboost",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["xgboost_252"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_wide_without_lag,
        "xgboost",
        window_size=252,
        period_of_forecast=90,
    )
)

trained_models_results["random_forest_100"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_wide_without_lag,
        "random_forest",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["random_forest_252"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_wide_without_lag,
        "random_forest",
        window_size=252,
        period_of_forecast=90,
    )
)

/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but Ra

In [16]:
fig = go.Figure()

for model_name, result in trained_models_results.items():
    fig.add_trace(
        go.Scatter(
            x=result.index,
            y=result["varForecast"],
            mode="lines",
            name=f"{model_name} VaR Forecast",
        )
    )

fig.add_trace(
    go.Scatter(
        x=trained_models_results["xgboost_100"].index,
        y=trained_models_results["xgboost_100"]["loss"],
        mode="lines",
        name="Loss",
    )
)

fig.update_layout(
    title="30-Day VaR Forecast and Loss",
    xaxis_title="Date",
    yaxis_title="Value",
    template="plotly_white",
)

fig.show()

Xgboost had done a terrible job for 252 day window, without hyperparameters + lagged, just assumed a constant value every time. Using lagged values:

In [15]:
vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_long_with_lag,
        "xgboost",
        window_size=100,
        period_of_forecast=30,
    )

,varForecast,loss
2024-11-18,0.010165,-0.005652
2024-11-19,0.008621,0.002235
2024-11-20,0.008690,-0.003572
2024-11-21,0.032407,-0.010010
2024-11-22,0.032247,-0.005688
2024-11-25,0.033740,-0.003076
2024-11-26,0.033866,0.001476
2024-11-27,0.033033,0.001440
2024-11-29,0.008446,-0.003649
2024-12-02,0.008352,0.003004


In [20]:
trained_models_results = {}

trained_models_results["xgboost_100"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_long_with_lag,
        "xgboost",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["xgboost_252"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_long_with_lag,
        "xgboost",
        window_size=252,
        period_of_forecast=90,
    )
)

trained_models_results["random_forest_100"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_long_with_lag,
        "random_forest",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["random_forest_252"] = (
    vlx.NON_ROLLING_TRAINER_single_model_multi_day_var(
        feature_long_with_lag,
        "random_forest",
        window_size=252,
        period_of_forecast=90,
    )
)

/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
/opt/homebrew/anaconda3/envs/hands_on_ml/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but Ra

In [21]:
fig = go.Figure()

for model_name, result in trained_models_results.items():
    fig.add_trace(
        go.Scatter(
            x=result.index,
            y=result["varForecast"],
            mode="lines",
            name=f"{model_name} VaR Forecast",
        )
    )

fig.add_trace(
    go.Scatter(
        x=trained_models_results["xgboost_100"].index,
        y=trained_models_results["xgboost_100"]["loss"],
        mode="lines",
        name="Loss",
    )
)

fig.update_layout(
    title="30-Day VaR Forecast and Loss",
    xaxis_title="Date",
    yaxis_title="Value",
    template="plotly_white",
)

fig.show()

## Rolling Trainer For Next Day
Testing the strategy, where we train the model everyday to predict

In [24]:
trained_models_results = {}

trained_models_results["GARCH_100"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_wide_without_lag, #garch just depends ron r
        "garch",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["GARCH_100"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_wide_without_lag, #garch just depends ron r
        "garch",
        window_size=252,
        period_of_forecast=90,
    )
)


trained_models_results["LSTM_100"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_long_with_lag,
        "lstm",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["LSTM_252"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_long_with_lag,
        "lstm",
        window_size=100,
        period_of_forecast=90,
    )
)


trained_models_results["xgboost_100"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_long_with_lag,
        "xgboost",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["xgboost_252"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_long_with_lag,
        "xgboost",
        window_size=252,
        period_of_forecast=90,
    )
)

trained_models_results["random_forest_100"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var_FOR_NEXT_DAY(
        feature_long_with_lag,
        "random_forest",
        window_size=100,
        period_of_forecast=90,
    )
)

trained_models_results["random_forest_252"] = (
    vlx.ROLLING_TRAINER_single_model_multi_day_var(
        feature_long_with_lag,
        "random_forest",
        window_size=252,
        period_of_forecast=90,
    )
)

KeyboardInterrupt: 

In [ ]:
fig = go.Figure()

for model_name, result in trained_models_results.items():
    fig.add_trace(
        go.Scatter(
            x=result.index,
            y=result["varForecast"],
            mode="lines",
            name=f"{model_name} VaR Forecast",
        )
    )

fig.add_trace(
    go.Scatter(
        x=trained_models_results["xgboost_100"].index,
        y=trained_models_results["xgboost_100"]["loss"],
        mode="lines",
        name="Loss",
    )
)

fig.update_layout(
    title="90-Day VaR Forecast and Loss",
    xaxis_title="Date",
    yaxis_title="Value",
    template="plotly_white",
)

fig.show()